In [1]:
from urllib.robotparser import RobotFileParser

def check_robots(url):
    """Check if scraping is allowed"""
    rp = RobotFileParser()
    rp.set_url('http://mtc-m16c.sid.inpe.br/robots.txt')
    rp.read()
    return rp.can_fetch('*', url)

if check_robots('http://mtc-m16c.sid.inpe.br/rep/8JMKD3MGP8W/3HD9RLP'):
    print("✅ Scraping allowed")
else:
    print("❌ Scraping blocked by robots.txt")

✅ Scraping allowed


In [ ]:
import json
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import csv
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

url_pagina_evento_07 = "http://mtc-m16c.sid.inpe.br/rep/8JMKD3MGP8W/3HD9RLP"
url_pagina_evento_15 = "http://mtc-m16c.sid.inpe.br/rep/8JMKD3MGPDW34P/42T2678"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
}

def descobrir_links_metadados(url_do_evento):
    print(f"\nDescobrindo links de artigos em: {url_do_evento}")
    links_descobertos = []
    
    try:
        response = requests.get(url_do_evento, headers=headers, timeout=30)
        response.raise_for_status()
        response.encoding = 'windows-1252' #acentuação correta
        soup = BeautifulSoup(response.text, "html.parser")
        
        frames = soup.find_all("frame", src=True)
        if frames:
            url_conteudo = None
            for f in frames:
                if f.get("name") != "header":  
                    url_conteudo = urljoin(url_do_evento, f["src"])
                    break
            
            if url_conteudo:
                response = requests.get(url_conteudo, headers=headers, timeout=30)
                response.raise_for_status()
                response.encoding = 'windows-1252'
                soup = BeautifulSoup(response.text, "html.parser")
        
        all_links = soup.find_all("a", href=True)
        print(f"Total de links: {len(all_links)}")
        
        for tag_a in all_links:
            href = tag_a["href"]
            texto = tag_a.get_text(" ", strip=True).lower()
            
            if (
                "mirrorget.cgi" in href.lower() or 
                "metadata" in href.lower() or 
                "citar" in texto or 
                "cite" in texto or 
                "metadados" in texto
            ):
                url_completa = urljoin(url_do_evento, href)
                if url_completa not in links_descobertos:
                    links_descobertos.append(url_completa)
                    
        print(f"{len(links_descobertos)} artigos filtrados para processar.")
    except Exception as e:
        print(f"Erro ao listar os links: {e}")
        
    return links_descobertos

def get_field(soup, name):
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) != 2:
            continue
        key = cells[0].get_text(" ", strip=True)
        if key == name:
            return cells[1].get_text(" ", strip=True)
    return None

def raspar_evento(url_do_evento, nome_pasta_output, ano_evento, n_edicao_padrao):
    lista_links = descobrir_links_metadados(url_do_evento)
    
    if not lista_links:
        print(f"Nenhum link de metadados para {nome_pasta_output}. Pulando...")
        return

    print(f"extraindo de conteúdo para a pasta: {nome_pasta_output}...")
    
    artigos_lista = []
    pessoas_dict = {}
    autoria_lista = []
    
    id_artigo_count = 1
    id_pessoa_count = 1
    id_autoria_count = 1
    
    for idx, url_bruta in enumerate(lista_links):
        try:
            parsed_url = urlparse(url_bruta)
            query_params = parse_qs(parsed_url.query)
            params = {k: v[0] for k, v in query_params.items()}
            
            repo_id = params.get('metadatarepository')
            if not repo_id:
                continue
            
            params['choice'] = 'full'
            params['languagebutton'] = 'en'
            
            url_limpa = urljoin(url_bruta, parsed_url.path) + '?' + urlencode(params)
            
            response = requests.get(url_limpa, headers=headers, timeout=30)
            response.raise_for_status()
            response.encoding = 'windows-1252' # CORREÇÃO: Força codificação nativa para acentos corretos
            soup = BeautifulSoup(response.text, "html.parser")
            
            sub_frames = soup.find_all("frame", src=True)
            if sub_frames:
                url_sub_conteudo = None
                for sf in sub_frames:
                    if sf.get("name") not in ["header", "sidebar", "menu"]:  
                        url_sub_conteudo = urljoin(url_limpa, sf["src"])
                        break
                if url_sub_conteudo:
                    response = requests.get(url_sub_conteudo, headers=headers, timeout=30)
                    response.raise_for_status()
                    response.encoding = 'windows-1252'
                    soup = BeautifulSoup(response.text, "html.parser")
            
            title = get_field(soup, "Title")
            abstract = get_field(soup, "Abstract")
            article_type = get_field(soup, "Tertiary Type")
            language = get_field(soup, "Language")
            conference_name = get_field(soup, "Conference Name")
            author = get_field(soup, "Author")
            affiliations = get_field(soup, "Affiliation")
            pages = get_field(soup, "Number of Pages") or get_field(soup, "Page Count")
            
            if not title:
                continue
                
            edition = n_edicao_padrao
            if conference_name:
                m = re.search(r",\s*(\d+)\s*\(", conference_name)
                if m:
                    edition = int(m.group(1))
            
            if article_type == "Full paper": article_type = "full"
            elif article_type == "Short paper": article_type = "short"
            
            if language == "en": language = "english"
            elif language == "pt": language = "portugues" # CORREÇÃO: Alinhado com a string do professor
            
            artigos_lista.append({
                "id_artigo": id_artigo_count,
                "titulo": title,
                "abstract": abstract,
                "n_paginas": pages if pages else "NULL",
                "tipo_artigo": article_type,
                "lingua": language,
                "n_edicao": edition
            })
            
            raw_authors = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', author) if author else []
            raw_authors = [name.strip() for name in raw_authors]
            formatted_authors = []
            for aut in raw_authors:
                aut_clean = aut.strip()
                if ',' in aut_clean:
                    parts = aut_clean.split(',', 1)
                    if len(parts) == 2:
                        formatted_authors.append(f"{parts[1].strip()} {parts[0].strip()}")
                    else:
                        formatted_authors.append(aut_clean)
                else:
                    formatted_authors.append(aut_clean)
            
            raw_affiliations = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', affiliations) if affiliations else []
            formatted_affiliations = [aff.strip().replace('\ufffd', '–') for aff in raw_affiliations]
            
            for idx_a, nome_autor in enumerate(formatted_authors):
                inst = formatted_affiliations[idx_a] if idx_a < len(formatted_affiliations) else "Desconhecida"
                
                if nome_autor not in pessoas_dict:
                    pessoas_dict[nome_autor] = {
                        "id_pessoa": id_pessoa_count,
                        "nome": nome_autor,
                        "instituicao": inst
                    }
                    id_pessoa_current = id_pessoa_count
                    id_pessoa_count += 1
                else:
                    id_pessoa_current = pessoas_dict[nome_autor]["id_pessoa"]
                
                #TRUE para o primeiro autor (ordem_autoria == 1) e FALSE para os demais
                is_corresponding = "TRUE" if idx_a == 0 else "FALSE"
                
                autoria_lista.append({
                    "id_autoria": id_autoria_count,
                    "id_artigo": id_artigo_count,
                    "id_pessoa": id_pessoa_current,
                    "ordem_autoria": idx_a + 1,
                    "autor_correspondente": is_corresponding
                })
                id_autoria_count += 1
                
            id_artigo_count += 1
            print(f"Processado: {title[:40]}...")
        except Exception as e:
            print(f"Erro ao processar artigo: {e}")
            
    if artigos_lista:
        os.makedirs(nome_pasta_output, exist_ok=True)
        
        df_artigos = pd.DataFrame(artigos_lista)
        df_pessoas = pd.DataFrame(list(pessoas_dict.values()))
        df_autoria = pd.DataFrame(autoria_lista)
        df_evento = pd.DataFrame([{"id_evento": 1, "n_edicao": n_edicao_padrao, "ano": ano_evento, "local": "INPE"}])
        
        df_comite = pd.DataFrame(columns=["id_revisor", "id_pessoa", "n_edicao"])
        df_organizacao = pd.DataFrame(columns=["id_organizador", "id_pessoa", "funcao", "n_edicao"])
        
        df_artigos.to_csv(f"{nome_pasta_output}/artigos.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_pessoas.to_csv(f"{nome_pasta_output}/pessoas.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_autoria.to_csv(f"{nome_pasta_output}/autoria.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_evento.to_csv(f"{nome_pasta_output}/evento.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_comite.to_csv(f"{nome_pasta_output}/comite_cientifico.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_organizacao.to_csv(f"{nome_pasta_output}/organizacao.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        print(f"Tabelas relacionais salvas em'{nome_pasta_output}'")

raspar_evento(url_pagina_evento_07, "2005", 2005, 7)
raspar_evento(url_pagina_evento_15, "2014", 2014, 15)